In [1]:
import numpy as np
import jax
import h5py
import matplotlib.pyplot as plt

# ── klsurprise package ──────────────────────────────────────────────
import klsurprise as kls
from klsurprise import (
    run_nested_sampling,
    load_create_NS_file,
    KLD_numerical,
    SurpriseGauss,
    find_pval,
    sigma_discordance,
)

# ── local example modules (same directory as this notebook) ─────────
import SNIa_likelihood_pantheon as SNIap
import BAO_likelihood_DESI as BAO
import plot_tools

/home/prm/klsurprise/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


# Model Configuration 

In [2]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║  Pick one model: "owCDM", "wCDM", "oLCDM", "FLCDM"             ║
# ╚════════════════════════════════════════════════════════════════════╝
model_name = "owCDM"

# ── date tag for file names ─────────────────────────────────────────
date_id = "20260315"

# ── model-dependent settings ────────────────────────────────────────
if model_name == "owCDM":
    domain = np.array([[0.3, 1.0], [0.05, 1.0], [-1.0, 1.0], [-3.0, -0.4]])
    names  = ["h", "Om", "Ok", "w"]
    labels = [r"$h$", r"$\Omega_m$", r"$\Omega_k$", r"$w$"]
    logP_1 = BAO.logP_marg_owCDM          # BAO+BBN
    logP_2 = SNIap.logposterior_owCDM     # Pantheon+SH0ES

elif model_name == "wCDM":
    domain = np.array([[0.3, 1.0], [0.05, 1.0], [-3.0, -0.4]])
    names  = ["h", "Om", "w"]
    labels = [r"$h$", r"$\Omega_m$", r"$w$"]
    logP_1 = BAO.logP_marg_wCDM
    logP_2 = SNIap.logposterior_wCDM

elif model_name == "oLCDM":
    domain = np.array([[0.3, 1.0], [0.05, 1.0], [-1.0, 1.0]])
    names  = ["h", "Om", "Ok"]
    labels = [r"$h$", r"$\Omega_m$", r"$\Omega_k$"]
    logP_1 = BAO.logP_marg_oLCDM
    logP_2 = SNIap.logposterior_oLCDM

elif model_name == "FLCDM":
    domain = np.array([[0.3, 1.0], [0.05, 1.0]])
    names  = ["h", "Om"]
    labels = [r"$h$", r"$\Omega_m$"]
    logP_1 = BAO.logP_marg_FLCDM
    logP_2 = SNIap.logposterior_LCDM

else:
    raise ValueError(f"Unknown model: {model_name}")

ndim = len(names)

# ── data-space model and covariance for dataset 2 (Pantheon) ───────
data_2_model_fun = SNIap.model_SNIa
data_2_covariance = SNIap.covariance_pantheon
data_2_vector = SNIap.mu_SNIa

# ── JIT-compiled mock log-likelihood (for PPD nested sampling) ─────
@jax.jit
def logL_mock(theta, data):
    return logP_2(theta, mu_SNIa=data)

# ── file names for NS result caching ───────────────────────────────
d1_name = f"{date_id}_DESI+BBN_{model_name}.pkl"
d2_name = f"{date_id}_Pantheon+SH0ES_{model_name}.pkl"

print(f"Model: {model_name}  ({ndim}D)")
print(f"Parameters: {names}")
print(f"Domain:\n{domain}")

Model: owCDM  (4D)
Parameters: ['h', 'Om', 'Ok', 'w']
Domain:
[[ 0.3   1.  ]
 [ 0.05  1.  ]
 [-1.    1.  ]
 [-3.   -0.4 ]]


# Nested Sampling: Dataset 1 (BAO+BBN)

Load cached result or run dynesty. Uses `load_create_NS_file` which handles
try/load/run/save automatically.

In [3]:
res_1 = load_create_NS_file(d1_name, logP_1, ndim, domain)
eq_samples_1 = res_1.samples_equal()

print(f"\nDataset 1 (BAO+BBN): {eq_samples_1.shape[0]} equal-weight samples")
print(f"  Mean: {eq_samples_1.mean(axis=0)}")
print(f"  Std:  {eq_samples_1.std(axis=0)}")

----------------------------------------------------------------------
Loading posterior data
----------------------------------------------------------------------
Data loaded sucessfully!

Dataset 1 (BAO+BBN): 24347 equal-weight samples
  Mean: [ 0.68671099  0.26143757  0.10041107 -1.23330114]
  Std:  [0.04119326 0.02787611 0.08070745 0.27927219]


# Nested Sampling: Dataset 2 (Pantheon+SH0ES)

In [4]:
res_2 = load_create_NS_file(d2_name, logP_2, ndim, domain)
eq_samples_2 = res_2.samples_equal()

print(f"\nDataset 2 (Pantheon+SH0ES): {eq_samples_2.shape[0]} equal-weight samples")
print(f"  Mean: {eq_samples_2.mean(axis=0)}")
print(f"  Std:  {eq_samples_2.std(axis=0)}")

----------------------------------------------------------------------
Loading posterior data
----------------------------------------------------------------------
Data loaded sucessfully!

Dataset 2 (Pantheon+SH0ES): 24796 equal-weight samples
  Mean: [ 0.72884333  0.23463864  0.15792306 -1.16757556]
  Std:  [0.00304301 0.07238048 0.30767693 0.56135408]


#  Inspect Posteriors: Triangle Plot

In [7]:

# plot_tools.create_triangle_plot(eq_samples_1, chain_2=eq_samples_2, D1="BAO+BBN (D1)", D2="Pantheon+SH0ES (D2)", names=names,labels=labels,)

# Compute KLD (Numerical)

In [8]:
kld_value = KLD_numerical(
    res_2, logP_2,
    res_1, logP_1,
    domain=domain,
)
print(f"\nKLD(D2 || D1) = {kld_value:.2f} nats")

Processing batches:   0%|                                                                                                                                                                 | 0/25 [00:00<?, ?it/s]/home/prm/klsurprise/.venv/lib/python3.12/site-packages/jax/_src/numpy/array_methods.py:126: UserWarning: Explicitly requested dtype int64 requested in astype is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)
Processing batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:02<00:00, 10.93it/s]



KLD(D2 || D1) = 339.32 nats


# Gaussian KLD (Analytical Comparison)

In [14]:
sg = SurpriseGauss(eq_samples_2, eq_samples_1)
kld_gauss = sg.calculate_kld()

print(f"KLD numerical:  {kld_value:.2f} nats")
print(f"KLD Gaussian:   {kld_gauss[0]:.2f} nats")

KLD numerical:  338.24 nats
KLD Gaussian:   55.42 nats


# Full Surprise Pipeline
Run the end-to-end pipeline: PPD generation, KLD distribution, Surprise.

In [ ]:
Nkld = 100  # number of PPD samples for the KLD distribution
    
sup = kls.surprise_statistics(
    logP_1,
    data_2_model_fun,
    covariance_matrix_2=data_2_covariance,
    domain=domain,
    data_2=data_2_vector,
    data_1_name=d1_name,
    data_2_name=d2_name,
)

results = sup.surprise_function_call(
    Nkld=Nkld,
    result_path=f"{date_id}_Surprise_{model_name}.hdf5",
    n_jobs=10
)

Handling dataset 1...
______________________________________________________________________
----------------------------------------------------------------------
Loading posterior data
----------------------------------------------------------------------
Data loaded sucessfully!
----------------------------------------------------------------------
Loading posterior data
----------------------------------------------------------------------
Data loaded sucessfully!
Done!

Handling posterior predictive distribution PPD(D2|D1) ...
______________________________________________________________________
Evaluating theory from sample distribution p1


Generating theory vectors: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 2967.91it/s]


Sampling the Posterior Predictive Distribution...


Sampling PPD: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 128.23it/s]


Handling KLD distribution...
______________________________________________________________________


Iterating over the PPD:   0%|                                                                                                                                                                                                                         | 0/100 [00:00<?, ?it/s]An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled

# Analyze Results

In [ ]:
S      = results["S"]
p_val  = results["p_value"]
sigma  = results["sigma_discordance"]
kld_d  = results["kld_dist"]

print(f"{'─'*50}")
print(f"  Model:              {model_name}")
print(f"  KLD(D2 || D1):      {results['kld21']:.2f} nats")
print(f"  <KLD>_PPD:          {results['kld_exp']:.2f} nats")
print(f"  S = KLD - <KLD>:    {S:.2f} nats")
print(f"  p-value:            {p_val:.4f}")
print(f"  Discordance:        {sigma:.2f} σ")
print(f"{'─'*50}")

# ── Gaussian Surprise for comparison ────────────────────────────────
sg_result = sg.calculate_surprise()
print(f"\n  Gaussian S:         {sg_result['S']:.2f} nats")
print(f"  Gaussian p-value:   {sg_result['p_value']:.4f}")

# Surprise Distribution Histogram

In [ ]:
S_dist = results["S_dist"]

# also compute Gaussian Surprise distribution
sg_full = SurpriseGauss(eq_samples_2, eq_samples_1)
sg_surprise = sg_full.calculate_surprise(Nsamples=len(S_dist))

plot_tools.plot_histograms(
    data_arrays=[S_dist, sg_surprise["S_dist"]],
    data_labels=["Full numerical", "Gaussian"],
    xlabel="S (nats)",
    title=f"Surprise distribution — {model_name}",
)

# mark the observed S values
plt.axvline(S, color="C0", ls="--", label=f"S = {S:.1f}")
plt.axvline(sg_surprise["S"], color="C1", ls="--", label=f"S (Gauss) = {sg_surprise['S']:.1f}")
plt.legend()
plt.tight_layout()
plt.show()